# GAM-Style Spline Tuning

This notebook extends the first additive spline baseline with controlled tuning over target definition, spline complexity, Ridge regularization, and feature-family choices. It remains intentionally notebook-local so the winning logic can be copied into a Kaggle submission notebook without relying on repository-only feature builders.

In [1]:
"Configure imports, paths, and deterministic settings."

from pathlib import Path
import json
import math
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = next(parent for parent in Path.cwd().parents if (parent / "src").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.validation import group_well_folds
from src.io.loaders import discover_wells, load_horizontal_well, load_typewell
from src.utils.run_bundle import RunBundle

warnings.filterwarnings("ignore", category=RuntimeWarning)

RUN_TAG = "gam_spline_tuning"
RUN_DATE = "2026-06-21"
SEED = 42
N_SPLITS = 3
MAX_TARGET_ROWS_PER_WELL = 140
MAX_TRAIN_ROWS_PER_FOLD = 80000

DATA_DIR = ROOT / "data" / "raw"
TRAIN_DIR = DATA_DIR / "train"
ARTIFACT_DIR = ROOT / "artifacts"
TABLE_DIR = ARTIFACT_DIR / "tables"
FIGURE_DIR = ARTIFACT_DIR / "figures"
REPORT_DIR = ARTIFACT_DIR / "reports"
MODEL_DIR = ARTIFACT_DIR / "models" / "gam"

for directory in [TABLE_DIR, FIGURE_DIR, REPORT_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
plt.style.use("seaborn-v0_8-whitegrid")


In [2]:
"Define inline feature construction helpers."


def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce").astype(float)


def clean_gr(series):
    gr = safe_numeric(series).clip(lower=0, upper=300)
    return gr.fillna(gr.rolling(21, center=True, min_periods=1).median()).ffill().bfill()


def deterministic_sample_positions(mask, max_rows):
    positions = np.flatnonzero(mask.to_numpy())
    if len(positions) <= max_rows:
        return positions
    return positions[np.linspace(0, len(positions) - 1, max_rows).astype(int)]


def window_slope(md, values, end_pos, window):
    start_pos = max(0, end_pos - window + 1)
    x0 = float(md.iloc[start_pos])
    x1 = float(md.iloc[end_pos])
    y0 = float(values.iloc[start_pos])
    y1 = float(values.iloc[end_pos])
    if not np.isfinite([x0, x1, y0, y1]).all() or abs(x1 - x0) < 1e-9:
        return np.nan
    return (y1 - y0) / (x1 - x0)


def typewell_interpolator(typewell):
    if typewell.empty or "TVT" not in typewell.columns or "GR" not in typewell.columns:
        return None
    tw = typewell[["TVT", "GR"]].dropna().sort_values("TVT")
    if len(tw) < 2:
        return None
    return interp1d(
        tw["TVT"].to_numpy(float),
        tw["GR"].to_numpy(float),
        bounds_error=False,
        fill_value=(float(tw["GR"].iloc[0]), float(tw["GR"].iloc[-1])),
    )


def known_typewell_corr(horizontal, typewell):
    interp = typewell_interpolator(typewell)
    if interp is None:
        return np.nan
    known = horizontal["TVT_input"].notna() & horizontal["GR"].notna()
    if int(known.sum()) < 20:
        return np.nan
    sample_pos = deterministic_sample_positions(known, 500)
    sample = horizontal.iloc[sample_pos]
    tw_gr = interp(sample["TVT_input"].to_numpy(float))
    if np.nanstd(tw_gr) < 1e-9 or np.nanstd(sample["GR"].to_numpy(float)) < 1e-9:
        return np.nan
    return float(np.corrcoef(sample["GR"].to_numpy(float), tw_gr)[0, 1])


def build_target_zone_features_for_well(well, directory, max_rows):
    horizontal = load_horizontal_well(directory / f"{well}__horizontal_well.csv")
    typewell_path = directory / f"{well}__typewell.csv"
    typewell = load_typewell(typewell_path) if typewell_path.exists() else pd.DataFrame()

    target_mask = horizontal["TVT_input"].isna() & horizontal["TVT"].notna()
    known_mask = horizontal["TVT_input"].notna() & horizontal["TVT"].notna()
    if int(target_mask.sum()) == 0 or int(known_mask.sum()) == 0:
        return pd.DataFrame()

    tvt_input = safe_numeric(horizontal["TVT_input"])
    md = safe_numeric(horizontal["MD"])
    x = safe_numeric(horizontal["X"])
    y = safe_numeric(horizontal["Y"])
    z = safe_numeric(horizontal["Z"])
    gr_raw = safe_numeric(horizontal["GR"])
    gr_imputed = clean_gr(horizontal["GR"])
    gr_missing = gr_raw.isna().astype(float)

    known_positions = np.flatnonzero(known_mask.to_numpy())
    last_known_pos = int(known_positions[-1])
    last_known_tvt = float(tvt_input.iloc[last_known_pos])
    last_known_md = float(md.iloc[last_known_pos])
    last_known_z = float(z.iloc[last_known_pos])
    last_known_gr = float(gr_imputed.iloc[last_known_pos])
    last_known_x = float(x.iloc[last_known_pos])
    last_known_y = float(y.iloc[last_known_pos])

    slope_windows = [10, 25, 50, 100]
    tvt_slopes = {f"tvt_slope_last_{w}": window_slope(md, tvt_input, last_known_pos, w) for w in slope_windows}
    z_slopes = {f"z_slope_last_{w}": window_slope(md, z, last_known_pos, w) for w in slope_windows}

    tw_interp = typewell_interpolator(typewell)
    typewell_corr = known_typewell_corr(horizontal, typewell)
    horizontal_step = np.sqrt(x.diff().fillna(0.0) ** 2 + y.diff().fillna(0.0) ** 2)
    horizontal_dist = horizontal_step.cumsum()
    gr_roll_mean_25 = gr_imputed.rolling(25, center=True, min_periods=1).mean()
    gr_roll_std_25 = gr_imputed.rolling(25, center=True, min_periods=2).std()
    gr_missing_frac_25 = gr_missing.rolling(25, center=True, min_periods=1).mean()
    gr_missing_frac_100 = gr_missing.rolling(100, center=True, min_periods=1).mean()

    target_positions = deterministic_sample_positions(target_mask, max_rows)
    all_target_positions = np.flatnonzero(target_mask.to_numpy())
    eval_rank = {int(pos): rank for rank, pos in enumerate(all_target_positions)}
    eval_denominator = max(len(all_target_positions) - 1, 1)
    rows = []
    for pos in target_positions:
        distance_md = float(md.iloc[pos] - last_known_md)
        slope50 = tvt_slopes["tvt_slope_last_50"]
        slope50_delta = 0.0 if not np.isfinite(slope50) else slope50 * distance_md
        tvt_proxy = last_known_tvt + slope50_delta
        tw_gr_proxy = float(tw_interp([tvt_proxy])[0]) if tw_interp is not None and np.isfinite(tvt_proxy) else np.nan
        row = {
            "wellname": well,
            "row_idx": int(horizontal.index[pos]),
            "target_rows_in_well": int(target_mask.sum()),
            "MD": float(md.iloc[pos]),
            "Z": float(z.iloc[pos]),
            "md_norm": float((md.iloc[pos] - md.min()) / (md.max() - md.min() + 1e-9)),
            "distance_from_boundary_md": distance_md,
            "eval_progress": float(eval_rank[int(pos)] / eval_denominator),
            "last_known_tvt": last_known_tvt,
            "last_known_md": last_known_md,
            "last_known_z": last_known_z,
            "last_known_gr": last_known_gr,
            "dX_from_boundary": float(x.iloc[pos] - last_known_x),
            "dY_from_boundary": float(y.iloc[pos] - last_known_y),
            "dZ_from_boundary": float(z.iloc[pos] - last_known_z),
            "horizontal_dist_from_boundary": float(horizontal_dist.iloc[pos] - horizontal_dist.iloc[last_known_pos]),
            "GR_imputed": float(gr_imputed.iloc[pos]),
            "GR_missing": float(gr_missing.iloc[pos]),
            "gr_roll_mean_25": float(gr_roll_mean_25.iloc[pos]),
            "gr_roll_std_25": float(gr_roll_std_25.iloc[pos]) if np.isfinite(gr_roll_std_25.iloc[pos]) else np.nan,
            "gr_missing_frac_25": float(gr_missing_frac_25.iloc[pos]),
            "gr_missing_frac_100": float(gr_missing_frac_100.iloc[pos]),
            "tw_gr_at_proxy": tw_gr_proxy,
            "gr_minus_tw_proxy": float(gr_imputed.iloc[pos] - tw_gr_proxy) if np.isfinite(tw_gr_proxy) else np.nan,
            "typewell_gr_corr": typewell_corr,
            "TVT": float(horizontal["TVT"].iloc[pos]),
        }
        for key, value in tvt_slopes.items():
            row[key] = value
            row[f"{key}_projected_delta"] = value * distance_md if np.isfinite(value) else np.nan
        for key, value in z_slopes.items():
            row[key] = value
        row["delta_tvt_from_boundary"] = row["TVT"] - last_known_tvt
        row["endpoint_hold_pred"] = last_known_tvt
        row["slope50_pred"] = last_known_tvt + slope50_delta
        rows.append(row)
    return pd.DataFrame(rows)


In [3]:
"Build the sampled target-zone modeling frame."

start = time.time()
train_wells = discover_wells(TRAIN_DIR)
frames = [build_target_zone_features_for_well(well, TRAIN_DIR, MAX_TARGET_ROWS_PER_WELL) for well in train_wells]
feature_frame = pd.concat([frame for frame in frames if not frame.empty], ignore_index=True)
build_seconds = time.time() - start

print({
    "wells": len(train_wells),
    "sample_rows": len(feature_frame),
    "build_seconds": round(build_seconds, 2),
})
feature_frame.head()

20:32:48 | INFO     | src.io.loaders | Discovered 773 wells in C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\data\raw\train


{'wells': 773, 'sample_rows': 108220, 'build_seconds': 122.49}


,wellname,row_idx,target_rows_in_well,MD,Z,md_norm,distance_from_boundary_md,eval_progress,last_known_tvt,last_known_md,...,tvt_slope_last_50_projected_delta,tvt_slope_last_100,tvt_slope_last_100_projected_delta,z_slope_last_10,z_slope_last_25,z_slope_last_50,z_slope_last_100,delta_tvt_from_boundary,endpoint_hold_pred,slope50_pred
0,000d7d20,1442,3836,12909.0,-9735.07,0.273261,1.0,0.000000,11747.37,12908.0,...,0.016327,0.008485,0.008485,0.007778,0.005833,0.002653,0.004343,0.01,11747.37,11747.386327
1,000d7d20,1469,3836,12936.0,-9734.69,0.278378,28.0,0.007040,11747.37,12908.0,...,0.457143,0.008485,0.237576,0.007778,0.005833,0.002653,0.004343,0.15,11747.37,11747.827143
2,000d7d20,1497,3836,12964.0,-9734.06,0.283684,56.0,0.014342,11747.37,12908.0,...,0.914286,0.008485,0.475152,0.007778,0.005833,0.002653,0.004343,0.09,11747.37,11748.284286
3,000d7d20,1524,3836,12991.0,-9733.28,0.288800,83.0,0.021382,11747.37,12908.0,...,1.355102,0.008485,0.704242,0.007778,0.005833,0.002653,0.004343,0.68,11747.37,11748.725102
4,000d7d20,1552,3836,13019.0,-9732.40,0.294106,111.0,0.028683,11747.37,12908.0,...,1.812245,0.008485,0.941818,0.007778,0.005833,0.002653,0.004343,1.29,11747.37,11749.182245


## Tuning Design

A GAM models the response as a sum of smooth one-dimensional functions plus optional linear effects. Here, `SplineTransformer` expands each smooth feature into basis functions and Ridge controls the wiggliness through shrinkage. The tuning space is deliberately structured: first tune the previously winning slope-rich feature family, then test a small number of feature-family additions under the best target/complexity region.

In [4]:
"Define feature families and candidate GAM configurations."

endpoint_features = ["last_known_tvt", "distance_from_boundary_md", "eval_progress"]
trajectory_features = ["MD", "Z", "md_norm", "dX_from_boundary", "dY_from_boundary", "dZ_from_boundary", "horizontal_dist_from_boundary"]
slope_features = ["tvt_slope_last_10", "tvt_slope_last_25", "tvt_slope_last_50", "tvt_slope_last_100", "z_slope_last_25", "z_slope_last_50", "z_slope_last_100"]
slope_projection_features = ["tvt_slope_last_10_projected_delta", "tvt_slope_last_25_projected_delta", "tvt_slope_last_50_projected_delta", "tvt_slope_last_100_projected_delta"]
gr_features = ["GR_imputed", "GR_missing", "last_known_gr", "gr_roll_mean_25", "gr_roll_std_25", "gr_missing_frac_25", "gr_missing_frac_100"]
typewell_features = ["tw_gr_at_proxy", "gr_minus_tw_proxy", "typewell_gr_corr"]

feature_specs = {
    "endpoint_boundary": {
        "smooth": ["distance_from_boundary_md", "eval_progress"],
        "linear": ["last_known_tvt"],
    },
    "slopes_core": {
        "smooth": ["distance_from_boundary_md", "eval_progress", "md_norm", "dZ_from_boundary", "horizontal_dist_from_boundary"] + slope_features + slope_projection_features,
        "linear": ["last_known_tvt"],
    },
    "slopes_plus_gr": {
        "smooth": ["distance_from_boundary_md", "eval_progress", "md_norm", "dZ_from_boundary", "horizontal_dist_from_boundary"] + slope_features + slope_projection_features + ["GR_imputed", "gr_roll_mean_25", "gr_roll_std_25", "gr_missing_frac_25"],
        "linear": ["last_known_tvt", "last_known_gr", "GR_missing"],
    },
    "slopes_plus_typewell": {
        "smooth": ["distance_from_boundary_md", "eval_progress", "md_norm", "dZ_from_boundary", "horizontal_dist_from_boundary"] + slope_features + slope_projection_features + ["tw_gr_at_proxy", "gr_minus_tw_proxy"],
        "linear": ["last_known_tvt", "typewell_gr_corr"],
    },
    "full_controlled": {
        "smooth": ["distance_from_boundary_md", "eval_progress", "md_norm", "dZ_from_boundary", "horizontal_dist_from_boundary"] + slope_features + slope_projection_features + ["GR_imputed", "gr_roll_mean_25", "gr_missing_frac_25", "tw_gr_at_proxy", "gr_minus_tw_proxy"],
        "linear": ["last_known_tvt", "last_known_gr", "GR_missing", "typewell_gr_corr"],
    },
}
feature_specs = {
    name: {**spec, "features": list(dict.fromkeys(spec["smooth"] + spec["linear"]))}
    for name, spec in feature_specs.items()
}

candidate_specs = []
for target_mode in ["delta_boundary", "residual_endpoint"]:
    for n_knots in [5, 7, 9]:
        for alpha in [10.0, 30.0, 100.0]:
            candidate_specs.append({
                "candidate": f"core__{target_mode}__k{n_knots}__d3__a{alpha:g}",
                "feature_family": "slopes_core",
                "target_mode": target_mode,
                "n_knots": n_knots,
                "degree": 3,
                "alpha": alpha,
            })
for n_knots in [6, 7, 8]:
    for alpha in [60.0, 100.0, 150.0, 220.0, 300.0]:
        candidate_specs.append({
            "candidate": f"core_fine__delta_boundary__k{n_knots}__d3__a{alpha:g}",
            "feature_family": "slopes_core",
            "target_mode": "delta_boundary",
            "n_knots": n_knots,
            "degree": 3,
            "alpha": alpha,
        })
for target_mode in ["delta_boundary", "residual_endpoint"]:
    for n_knots in [7, 9]:
        candidate_specs.append({
            "candidate": f"core__{target_mode}__k{n_knots}__d2__a30",
            "feature_family": "slopes_core",
            "target_mode": target_mode,
            "n_knots": n_knots,
            "degree": 2,
            "alpha": 30.0,
        })
for feature_family in ["endpoint_boundary", "slopes_plus_gr", "slopes_plus_typewell", "full_controlled"]:
    for alpha in [30.0, 100.0]:
        candidate_specs.append({
            "candidate": f"{feature_family}__residual_endpoint__k7__d3__a{alpha:g}",
            "feature_family": feature_family,
            "target_mode": "residual_endpoint",
            "n_knots": 7,
            "degree": 3,
            "alpha": alpha,
        })

candidate_table = pd.DataFrame(candidate_specs)
candidate_table["n_smooth_features"] = candidate_table["feature_family"].map(lambda name: len(feature_specs[name]["smooth"]))
candidate_table["n_linear_features"] = candidate_table["feature_family"].map(lambda name: len(feature_specs[name]["linear"]))
candidate_table["n_total_features"] = candidate_table["feature_family"].map(lambda name: len(feature_specs[name]["features"]))
print({"candidate_count": len(candidate_table)})
candidate_table.head(10)

{'candidate_count': 45}


,candidate,feature_family,target_mode,n_knots,degree,alpha,n_smooth_features,n_linear_features,n_total_features
0,core__delta_boundary__k5__d3__a10,slopes_core,delta_boundary,5,3,10.0,16,1,17
1,core__delta_boundary__k5__d3__a30,slopes_core,delta_boundary,5,3,30.0,16,1,17
2,core__delta_boundary__k5__d3__a100,slopes_core,delta_boundary,5,3,100.0,16,1,17
3,core__delta_boundary__k7__d3__a10,slopes_core,delta_boundary,7,3,10.0,16,1,17
4,core__delta_boundary__k7__d3__a30,slopes_core,delta_boundary,7,3,30.0,16,1,17
5,core__delta_boundary__k7__d3__a100,slopes_core,delta_boundary,7,3,100.0,16,1,17
6,core__delta_boundary__k9__d3__a10,slopes_core,delta_boundary,9,3,10.0,16,1,17
7,core__delta_boundary__k9__d3__a30,slopes_core,delta_boundary,9,3,30.0,16,1,17
8,core__delta_boundary__k9__d3__a100,slopes_core,delta_boundary,9,3,100.0,16,1,17
9,core__residual_endpoint__k5__d3__a10,slopes_core,residual_endpoint,5,3,10.0,16,1,17


In [5]:
"Define additive spline fitting, target, and validation helpers."


def make_additive_spline_model(smooth_features, linear_features, n_knots, degree, alpha):
    transformers = []
    if smooth_features:
        transformers.append((
            "smooth",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("splines", SplineTransformer(n_knots=n_knots, degree=degree, include_bias=False)),
            ]),
            smooth_features,
        ))
    if linear_features:
        transformers.append((
            "linear",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            linear_features,
        ))
    return Pipeline([
        ("features", ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.0)),
        ("ridge", Ridge(alpha=alpha)),
    ])


def deterministic_row_cap(frame, max_rows):
    if len(frame) <= max_rows:
        return frame
    positions = np.linspace(0, len(frame) - 1, max_rows).astype(int)
    return frame.iloc[positions].copy()


def target_values(frame, target_mode):
    if target_mode == "delta_boundary":
        return frame["delta_tvt_from_boundary"].to_numpy(float)
    if target_mode == "residual_endpoint":
        return (frame["TVT"].to_numpy(float) - frame["endpoint_hold_pred"].to_numpy(float))
    if target_mode == "raw_tvt":
        return frame["TVT"].to_numpy(float)
    raise ValueError(f"Unknown target mode: {target_mode}")


def reconstruct_tvt(frame, prediction, target_mode):
    if target_mode == "delta_boundary":
        return frame["last_known_tvt"].to_numpy(float) + prediction
    if target_mode == "residual_endpoint":
        return frame["endpoint_hold_pred"].to_numpy(float) + prediction
    if target_mode == "raw_tvt":
        return prediction
    raise ValueError(f"Unknown target mode: {target_mode}")


def metric_row(y_true, pred_tvt):
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, pred_tvt))),
        "mae": float(mean_absolute_error(y_true, pred_tvt)),
        "r2": float(r2_score(y_true, pred_tvt)),
    }


def evaluate_endpoint(valid_frame):
    return valid_frame["endpoint_hold_pred"].to_numpy(float)


def evaluate_slope50(valid_frame):
    return valid_frame["slope50_pred"].to_numpy(float)

In [6]:
"Run grouped well-fold validation for the tuning candidates."

fit_start = time.time()
fold_rows = []
residual_rows = []
wells = sorted(feature_frame["wellname"].unique())
folds = group_well_folds(wells, n_splits=N_SPLITS)
sample_per_candidate_fold = 700

for fold_idx, (train_fold_wells, valid_fold_wells) in enumerate(folds, start=1):
    train_frame = feature_frame.loc[feature_frame["wellname"].isin(train_fold_wells)].copy()
    valid_frame = feature_frame.loc[feature_frame["wellname"].isin(valid_fold_wells)].copy()
    train_frame = deterministic_row_cap(train_frame, MAX_TRAIN_ROWS_PER_FOLD)
    y_true = valid_frame["TVT"].to_numpy(float)

    for baseline_name, pred in [("endpoint_hold", evaluate_endpoint(valid_frame)), ("slope50_projection", evaluate_slope50(valid_frame))]:
        metrics = metric_row(y_true, pred)
        fold_rows.append({
            "fold": fold_idx,
            "candidate": baseline_name,
            "model_family": "baseline",
            "feature_family": "baseline",
            "target_mode": "baseline",
            "n_knots": 0,
            "degree": 0,
            "alpha": 0.0,
            "n_train_rows": 0,
            "n_valid_rows": len(valid_frame),
            **metrics,
        })

    for spec in candidate_specs:
        family = feature_specs[spec["feature_family"]]
        model = make_additive_spline_model(family["smooth"], family["linear"], spec["n_knots"], spec["degree"], spec["alpha"])
        y_train = target_values(train_frame, spec["target_mode"])
        model.fit(train_frame[family["features"]], y_train)
        pred_target = model.predict(valid_frame[family["features"]])
        pred_tvt = reconstruct_tvt(valid_frame, pred_target, spec["target_mode"])
        metrics = metric_row(y_true, pred_tvt)
        fold_rows.append({
            "fold": fold_idx,
            "candidate": spec["candidate"],
            "model_family": "gam_spline_tuning",
            "feature_family": spec["feature_family"],
            "target_mode": spec["target_mode"],
            "n_knots": spec["n_knots"],
            "degree": spec["degree"],
            "alpha": spec["alpha"],
            "n_train_rows": len(train_frame),
            "n_valid_rows": len(valid_frame),
            **metrics,
        })
        sample_idx = np.linspace(0, len(valid_frame) - 1, min(len(valid_frame), sample_per_candidate_fold)).astype(int)
        residual_rows.append(pd.DataFrame({
            "fold": fold_idx,
            "candidate": spec["candidate"],
            "feature_family": spec["feature_family"],
            "target_mode": spec["target_mode"],
            "wellname": valid_frame.iloc[sample_idx]["wellname"].to_numpy(),
            "row_idx": valid_frame.iloc[sample_idx]["row_idx"].to_numpy(),
            "distance_from_boundary_md": valid_frame.iloc[sample_idx]["distance_from_boundary_md"].to_numpy(float),
            "eval_progress": valid_frame.iloc[sample_idx]["eval_progress"].to_numpy(float),
            "y_true": y_true[sample_idx],
            "y_pred": pred_tvt[sample_idx],
            "residual": y_true[sample_idx] - pred_tvt[sample_idx],
        }))

fit_seconds = time.time() - fit_start
fold_metrics = pd.DataFrame(fold_rows)
summary = (
    fold_metrics.groupby(["model_family", "candidate", "feature_family", "target_mode", "n_knots", "degree", "alpha"], as_index=False)
    .agg(rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"), mae_mean=("mae", "mean"), r2_mean=("r2", "mean"), n_valid_rows=("n_valid_rows", "sum"), n_train_rows_mean=("n_train_rows", "mean"))
    .sort_values("rmse_mean")
)
residual_sample = pd.concat(residual_rows, ignore_index=True)
print({"fit_seconds": round(fit_seconds, 2), "candidates": len(candidate_specs), "fold_metric_rows": len(fold_metrics)})
summary.head(12)

{'fit_seconds': 150.9, 'candidates': 45, 'fold_metric_rows': 141}


,model_family,candidate,feature_family,target_mode,n_knots,degree,alpha,rmse_mean,rmse_std,mae_mean,r2_mean,n_valid_rows,n_train_rows_mean
33,gam_spline_tuning,core_fine__delta_boundary__k7__d3__a60,slopes_core,delta_boundary,7,3,60.0,14.868732,0.501206,10.482774,0.999423,108220,72146.666667
7,gam_spline_tuning,core__delta_boundary__k7__d3__a100,slopes_core,delta_boundary,7,3,100.0,14.871530,0.479493,10.481390,0.999423,108220,72146.666667
29,gam_spline_tuning,core_fine__delta_boundary__k7__d3__a100,slopes_core,delta_boundary,7,3,100.0,14.871530,0.479493,10.481390,0.999423,108220,72146.666667
18,gam_spline_tuning,core__residual_endpoint__k7__d3__a100,slopes_core,residual_endpoint,7,3,100.0,14.871530,0.479493,10.481390,0.999423,108220,72146.666667
19,gam_spline_tuning,core__residual_endpoint__k7__d3__a30,slopes_core,residual_endpoint,7,3,30.0,14.871892,0.533661,10.489798,0.999423,108220,72146.666667
8,gam_spline_tuning,core__delta_boundary__k7__d3__a30,slopes_core,delta_boundary,7,3,30.0,14.871892,0.533661,10.489798,0.999423,108220,72146.666667
4,gam_spline_tuning,core__delta_boundary__k5__d3__a30,slopes_core,delta_boundary,5,3,30.0,14.875399,0.472231,10.480276,0.999423,108220,72146.666667
15,gam_spline_tuning,core__residual_endpoint__k5__d3__a30,slopes_core,residual_endpoint,5,3,30.0,14.875399,0.472231,10.480276,0.999423,108220,72146.666667
30,gam_spline_tuning,core_fine__delta_boundary__k7__d3__a150,slopes_core,delta_boundary,7,3,150.0,14.877285,0.465194,10.483347,0.999423,108220,72146.666667
28,gam_spline_tuning,core_fine__delta_boundary__k6__d3__a60,slopes_core,delta_boundary,6,3,60.0,14.879150,0.490102,10.480313,0.999423,108220,72146.666667


In [7]:
"Export tuning tables and diagnostic figures."

fold_metrics_path = TABLE_DIR / f"model__gam_spline_tuning_fold_metrics__{RUN_DATE}.csv"
summary_path = TABLE_DIR / f"model__gam_spline_tuning_summary__{RUN_DATE}.csv"
candidate_path = TABLE_DIR / f"model__gam_spline_tuning_candidates__{RUN_DATE}.csv"
residual_path = TABLE_DIR / f"model__gam_spline_tuning_residual_sample__{RUN_DATE}.csv"
fold_metrics.to_csv(fold_metrics_path, index=False)
summary.to_csv(summary_path, index=False)
candidate_table.to_csv(candidate_path, index=False)
residual_sample.to_csv(residual_path, index=False)

fig_paths = {}
gam_summary = summary.loc[summary["model_family"] == "gam_spline_tuning"].copy()
plot_summary = pd.concat([gam_summary.head(12), summary.loc[summary["candidate"] == "endpoint_hold"]], ignore_index=True)
plot_summary = plot_summary.sort_values("rmse_mean", ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#4C78A8" if family == "gam_spline_tuning" else "#F58518" for family in plot_summary["model_family"]]
ax.barh(plot_summary["candidate"], plot_summary["rmse_mean"], color=colors)
ax.set_title("Tuned GAM-Style Candidate RMSE")
ax.set_xlabel("grouped target-zone RMSE")
ax.set_ylabel("")
fig.tight_layout()
fig_paths["rmse_comparison"] = FIGURE_DIR / f"model__gam_spline_tuning_rmse_comparison__{RUN_DATE}.png"
fig.savefig(fig_paths["rmse_comparison"], dpi=160)
plt.close(fig)

best_candidate = str(gam_summary.iloc[0]["candidate"])
fig, ax = plt.subplots(figsize=(8, 5))
for candidate, group in fold_metrics.groupby("candidate"):
    if candidate in {"endpoint_hold", "slope50_projection", best_candidate} or candidate in set(gam_summary.head(3)["candidate"]):
        ax.plot(group["fold"], group["rmse"], marker="o", label=candidate)
ax.set_title("Tuned GAM Fold RMSE")
ax.set_xlabel("fold")
ax.set_ylabel("RMSE")
ax.legend(fontsize=7)
fig.tight_layout()
fig_paths["fold_rmse"] = FIGURE_DIR / f"model__gam_spline_tuning_fold_rmse__{RUN_DATE}.png"
fig.savefig(fig_paths["fold_rmse"], dpi=160)
plt.close(fig)

best_residuals = residual_sample.loc[residual_sample["candidate"] == best_candidate].copy()
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(best_residuals["y_pred"], best_residuals["residual"], s=6, alpha=0.35, color="#54A24B")
ax.axhline(0.0, color="black", linewidth=1)
ax.set_title(f"Residuals for {best_candidate}")
ax.set_xlabel("predicted TVT")
ax.set_ylabel("true - predicted")
fig.tight_layout()
fig_paths["best_residuals"] = FIGURE_DIR / f"model__gam_spline_tuning_best_residuals__{RUN_DATE}.png"
fig.savefig(fig_paths["best_residuals"], dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 5))
ax.hexbin(best_residuals["distance_from_boundary_md"], best_residuals["residual"], gridsize=45, mincnt=1, cmap="viridis")
ax.axhline(0.0, color="black", linewidth=1)
ax.set_title("Best Tuned GAM Residuals vs Boundary Distance")
ax.set_xlabel("MD distance past boundary")
ax.set_ylabel("true - predicted")
fig.tight_layout()
fig_paths["residual_by_boundary"] = FIGURE_DIR / f"model__gam_spline_tuning_residual_by_boundary__{RUN_DATE}.png"
fig.savefig(fig_paths["residual_by_boundary"], dpi=160)
plt.close(fig)

{key: str(value) for key, value in fig_paths.items()}

{'rmse_comparison': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__gam_spline_tuning_rmse_comparison__2026-06-21.png',
 'fold_rmse': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__gam_spline_tuning_fold_rmse__2026-06-21.png',
 'best_residuals': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__gam_spline_tuning_best_residuals__2026-06-21.png',
 'residual_by_boundary': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__gam_spline_tuning_residual_by_boundary__2026-06-21.png'}

In [8]:
"Write report, update comparison table, and create run bundle."

best_gam = gam_summary.iloc[0]
endpoint = summary.loc[summary["candidate"] == "endpoint_hold"].iloc[0]
slope50 = summary.loc[summary["candidate"] == "slope50_projection"].iloc[0]
phase1_best_rmse = 14.871891912103736
phase1_best_name = "boundary_trajectory_slopes"

family_means = (
    gam_summary.groupby("feature_family", as_index=False)
    .agg(best_rmse=("rmse_mean", "min"), median_rmse=("rmse_mean", "median"), candidate_count=("candidate", "count"))
    .sort_values("best_rmse")
)
target_means = (
    gam_summary.groupby("target_mode", as_index=False)
    .agg(best_rmse=("rmse_mean", "min"), median_rmse=("rmse_mean", "median"), candidate_count=("candidate", "count"))
    .sort_values("best_rmse")
)

findings = pd.DataFrame([
    {
        "topic": "best_tuned_gam",
        "finding": f"Best tuned candidate is {best_gam.candidate} with sampled grouped RMSE {best_gam.rmse_mean:.4f}.",
        "modeling_implication": "Use this candidate as the current GAM submission candidate if fold behavior is stable.",
    },
    {
        "topic": "phase1_comparison",
        "finding": f"Phase 2 best RMSE {best_gam.rmse_mean:.4f} vs Phase 1 best {phase1_best_name} RMSE {phase1_best_rmse:.4f}.",
        "modeling_implication": "Submit only if the tuned model improves materially and does not concentrate gains in one fold.",
    },
    {
        "topic": "endpoint_comparison",
        "finding": f"Best tuned GAM RMSE {best_gam.rmse_mean:.4f} vs endpoint-hold RMSE {endpoint.rmse_mean:.4f}.",
        "modeling_implication": "Endpoint hold remains the minimum useful local hurdle for this diagnostic protocol.",
    },
    {
        "topic": "target_mode_signal",
        "finding": f"Best target mode by RMSE is {target_means.iloc[0].target_mode}; feature family leader is {family_means.iloc[0].feature_family}.",
        "modeling_implication": "Carry these choices into a self-contained Kaggle notebook before trying more complex GAM libraries.",
    },
])
findings_path = TABLE_DIR / f"model__gam_spline_tuning_findings__{RUN_DATE}.csv"
family_path = TABLE_DIR / f"model__gam_spline_tuning_family_summary__{RUN_DATE}.csv"
target_path = TABLE_DIR / f"model__gam_spline_tuning_target_summary__{RUN_DATE}.csv"
findings.to_csv(findings_path, index=False)
family_means.to_csv(family_path, index=False)
target_means.to_csv(target_path, index=False)

report_path = REPORT_DIR / f"model__gam_spline_tuning__{RUN_DATE}.md"
report_lines = [
    "# GAM-Style Spline Tuning",
    "",
    f"Date: {RUN_DATE}",
    "",
    "## Key Findings",
]
for row in findings.itertuples(index=False):
    report_lines.extend([
        f"- **{row.topic}**: {row.finding}",
        f"  - Modeling implication: {row.modeling_implication}",
    ])
report_lines.extend([
    "",
    "## Best Candidate",
    f"- Candidate: `{best_gam.candidate}`",
    f"- Feature family: `{best_gam.feature_family}`",
    f"- Target mode: `{best_gam.target_mode}`",
    f"- Spline: knots={int(best_gam.n_knots)}, degree={int(best_gam.degree)}, alpha={float(best_gam.alpha):g}",
    f"- RMSE mean/std: {best_gam.rmse_mean:.4f} / {best_gam.rmse_std:.4f}",
    "",
    "## Artifacts",
    f"- Fold metrics: {fold_metrics_path}",
    f"- Summary: {summary_path}",
    f"- Candidate table: {candidate_path}",
    f"- Residual sample: {residual_path}",
    f"- Findings: {findings_path}",
    f"- Feature-family summary: {family_path}",
    f"- Target summary: {target_path}",
])
for name, fig_path in fig_paths.items():
    report_lines.append(f"- {name}: {fig_path}")
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

comparison_path = TABLE_DIR / "model_comparison.csv"
comparison = pd.read_csv(comparison_path)
run_comparison_row = pd.DataFrame([{
    "run_id": "pending_run_bundle",
    "model_family": "gam_spline_tuning",
    "model_name": str(best_gam.candidate),
    "cv_rmse_mean": float(best_gam.rmse_mean),
    "cv_rmse_std": float(best_gam.rmse_std),
    "train_time_s": float(build_seconds + fit_seconds),
    "n_features": int(candidate_table.loc[candidate_table["candidate"] == best_gam.candidate, "n_total_features"].iloc[0]),
    "notes": "Tuned GAM-style additive spline Ridge; sampled target-zone grouped CV; notebook-local Kaggle-safe features",
}])
comparison = comparison.loc[comparison["run_id"] != "pending_run_bundle"]
comparison = pd.concat([comparison, run_comparison_row], ignore_index=True)
comparison.to_csv(comparison_path, index=False)

config = {
    "task": "gam_spline_tuning",
    "seed": SEED,
    "n_splits": N_SPLITS,
    "max_target_rows_per_well": MAX_TARGET_ROWS_PER_WELL,
    "max_train_rows_per_fold": MAX_TRAIN_ROWS_PER_FOLD,
    "candidate_count": int(len(candidate_specs)),
    "best_candidate": str(best_gam.candidate),
    "best_feature_family": str(best_gam.feature_family),
    "best_target_mode": str(best_gam.target_mode),
    "best_n_knots": int(best_gam.n_knots),
    "best_degree": int(best_gam.degree),
    "best_alpha": float(best_gam.alpha),
    "note": "Notebook-local GAM-style spline tuning; target-zone diagnostic CV.",
}
with RunBundle(RUN_TAG, config, seed=SEED) as rb:
    rb.add_input(str(TRAIN_DIR))
    rb.add_metric("sample_rows", float(len(feature_frame)))
    rb.add_metric("candidate_count", float(len(candidate_specs)))
    rb.add_metric("best_gam_rmse", float(best_gam.rmse_mean))
    rb.add_metric("best_gam_rmse_std", float(best_gam.rmse_std))
    rb.add_metric("endpoint_hold_rmse", float(endpoint.rmse_mean))
    rb.add_metric("slope50_projection_rmse", float(slope50.rmse_mean))
    rb.add_metric("best_gam_minus_endpoint", float(best_gam.rmse_mean - endpoint.rmse_mean))
    rb.add_metric("best_gam_minus_phase1", float(best_gam.rmse_mean - phase1_best_rmse))
    for output_path in [fold_metrics_path, summary_path, candidate_path, residual_path, findings_path, family_path, target_path, report_path, comparison_path]:
        rb.add_output(str(output_path))
    for fig_path in fig_paths.values():
        rb.add_output(str(fig_path))
    rb.set_notes("GAM-style spline tuning completed. Candidate should be self-contained before any Kaggle submission.")
run_id = rb.run_id
comparison.loc[comparison["run_id"] == "pending_run_bundle", "run_id"] = run_id
comparison.to_csv(comparison_path, index=False)

print({
    "run_id": run_id,
    "best_candidate": best_gam.candidate,
    "best_rmse": round(float(best_gam.rmse_mean), 4),
    "endpoint_hold_rmse": round(float(endpoint.rmse_mean), 4),
    "phase1_delta": round(float(best_gam.rmse_mean - phase1_best_rmse), 4),
})
findings

{'run_id': '2026-06-21__2037__gam_spline_tuning__4e31e0e', 'best_candidate': 'core_fine__delta_boundary__k7__d3__a60', 'best_rmse': 14.8687, 'endpoint_hold_rmse': 15.5782, 'phase1_delta': -0.0032}


,topic,finding,modeling_implication
0,best_tuned_gam,Best tuned candidate is core_fine__delta_bound...,Use this candidate as the current GAM submissi...
1,phase1_comparison,Phase 2 best RMSE 14.8687 vs Phase 1 best boun...,Submit only if the tuned model improves materi...
2,endpoint_comparison,Best tuned GAM RMSE 14.8687 vs endpoint-hold R...,Endpoint hold remains the minimum useful local...
3,target_mode_signal,Best target mode by RMSE is delta_boundary; fe...,Carry these choices into a self-contained Kagg...


## Results

Review the executed outputs above for the tuned candidate ranking and fold stability. A candidate should only become a Kaggle submission notebook if it improves the first GAM baseline without relying on one anomalous fold.